# Laboratório: texto como dado

**A pergunta: do que reclamam os clientes da Olist?** A nota diz quanto o cliente ficou
insatisfeito; o comentário diz por quê. Neste laboratório transformamos os comentários em
tokens, contamos, e chegamos às expressões que marcam as avaliações ruins: a primeira resposta.

Escopo de hoje: este laboratório (texto como dado) e o próximo (higiene de corpus). **Na próxima
semana: representações vetoriais de texto (embeddings)**, que agrupam as avaliações pelo sentido.

Roteiro, na ordem dos slides: os dados → token e tipo → do texto ao byte → split e regex →
tokenizadores prontos → estatísticas → stopwords → Zipf → a primeira resposta.

## 0. Preparação

O notebook roda no **Google Colab** ou no seu computador. A primeira célula instala as bibliotecas
que faltam (rode uma vez); a de dados baixa as avaliações da Olist direto do Kaggle com `curl`, sem
conta. Se o download falhar, baixe o zip pelo navegador em
`kaggle.com/datasets/olistbr/brazilian-ecommerce` e extraia `olist_order_reviews_dataset.csv` na
pasta `../dados/olist/`.

In [ ]:
# Instala o que falta: no Colab, só as bibliotecas de texto; no seu computador, tudo
%pip install -q pandas numpy matplotlib regex tiktoken nltk spacy

import importlib, subprocess, sys
import nltk
nltk.download("punkt_tab", quiet=True)   # o tokenizador de sentenças do NLTK
nltk.download("stopwords", quiet=True)   # a lista de stopwords em português
try:
    import spacy
    spacy.load("pt_core_news_sm")
except OSError:                          # o modelo do spaCy em português, baixado uma vez
    subprocess.run([sys.executable, "-m", "spacy", "download", "pt_core_news_sm"], check=True)
    importlib.invalidate_caches()


In [ ]:
import re
import unicodedata
import zipfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

GRAY, RED, INDIGO, GREEN, LIGHT = "#64748b", "#ef4444", "#6366f1", "#22c55e", "#cbd5e1"
plt.rcParams.update({"figure.figsize": (9, 3.2), "font.size": 12, "axes.spines.top": False,
                     "axes.spines.right": False})


def br(valor, casas=0):
    # 40.977 e 14,0: milhar com ponto, decimal com vírgula
    return f"{valor:,.{casas}f}".replace(",", "§").replace(".", ",").replace("§", ".")


In [ ]:
URL = "https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce"
PASTA = Path("../dados/olist")  # Altere para o caminho do seu sistema, se já tiver os CSV
CAMINHO = PASTA / "olist_order_reviews_dataset.csv"

if not CAMINHO.exists():  # baixa o zip do Kaggle (cerca de 45 MB) e extrai só as avaliações
    PASTA.mkdir(parents=True, exist_ok=True)
    print("baixando do Kaggle...")
    !curl -L --fail -sS -o {PASTA}/olist.zip {URL}
    with zipfile.ZipFile(PASTA / "olist.zip") as arquivo:
        arquivo.extract(CAMINHO.name, PASTA)

bruto = pd.read_csv(CAMINHO, dtype=object)
avaliacoes = pd.DataFrame({"review_id": bruto["review_id"], "nota": bruto["review_score"].astype(int),
                           "mensagem": bruto["review_comment_message"],
                           "data": pd.to_datetime(bruto["review_creation_date"])})
mensagens = avaliacoes.dropna(subset=["mensagem"]).reset_index(drop=True)
mensagens["grupo"] = np.select([mensagens["nota"] <= 2, mensagens["nota"] == 3], ["ruim", "neutra"], "boa")
print(f"{br(len(avaliacoes))} avaliações; {br(len(mensagens))} com comentário")
mensagens["grupo"].value_counts()


In [ ]:
def grafico_ruins_boas(rotulos, ruins, boas, titulo, antes=None):
    # barras horizontais: % das avaliações ruins (vermelho) e das boas (cinza), como nos slides
    rotulos, ruins, boas = list(rotulos), list(ruins), list(boas)
    fig, ax = plt.subplots(figsize=(9, 0.5 * len(rotulos) + 1.2))
    y = np.arange(len(rotulos))[::-1]
    ax.barh(y + 0.18, ruins, height=0.36, color=RED, label="ruins (notas 1 e 2)")
    ax.barh(y - 0.18, boas, height=0.36, color=LIGHT, label="boas (notas 4 e 5)")
    if antes is not None:
        ax.scatter(list(antes), y + 0.18, marker="|", s=400, color="black", label="ruins, antes da higiene")
    for yi, v in zip(y, ruins):
        ax.text(v + 0.3, yi + 0.18, br(v, 1) + "%", va="center", fontsize=11)
    ax.set_yticks(y, rotulos)
    ax.set_xlabel("% das avaliações")
    ax.set_title(titulo)
    ax.legend(frameon=False, loc="lower right")
    plt.tight_layout()
    plt.show()


## 1. Quem escreve

Quem avalia mal escreve mais? A proporção de avaliações com comentário, por nota.

In [ ]:
# por nota: a % das avaliações que têm comentário (mensagem não nula)

# barras por nota: 1 e 2 em vermelho, 3 em cinza, 4 e 5 em verde

ax.set_ylabel("% com comentário")
ax.set_title("quem avalia mal escreve mais")
plt.show()
taxa.round(1)


O texto como está no arquivo: `repr` mostra os espaços sobrando e as quebras de linha (`\r\n`)
que a tela esconde.

In [ ]:
EXEMPLOS = [69, 29336, 279, 23817]  # posições de quatro avaliações em mensagens
for i in EXEMPLOS:
    # o texto cru, com repr


## 2. Corpus, token e tipo

**Corpus** é o conjunto de textos analisado: a coluna `mensagem`. **Token** é cada pedaço em que
o texto é cortado para ser contado. **Tipo** é um token único: as repetições contam uma vez só;
os tipos do corpus formam o **vocabulário**. Começamos com um regex simples: sequências de letras
ou dígitos.

In [ ]:
frase = "Até agora não recebi nada, a colcha estava prevista para anteontem(18/12) e até agora nada"
# os tokens: cada sequência de letras ou dígitos (\w+)
# os tipos: os tokens únicos, em minúsculas, com a contagem de cada um

print(len(tokens), "tokens:", tokens)
print(len(tipos), "tipos (tokens únicos):", tipos.most_common(4))


## 3. Da sentença ao byte

**Unicode** é o padrão que dá um número a cada caractere de todas as escritas, emojis incluídos:
o **ponto de código**, escrito U+ e o número em hexadecimal (é = U+00E9). O UTF-8 grava cada
número em 1 a 4 bytes. **Grafema** é o caractere que o leitor vê na tela; pode ser feito de mais
de um ponto de código: 👍🏻 é o polegar (U+1F44D) mais o tom de pele (U+1F3FB).

In [ ]:
import regex  # o módulo regex entende \X (grafema)

frase = "Produto bem acabado 👍🏻"
# os grafemas: o que o leitor vê (\X do módulo regex)

# os pontos de código: ord dá o número; U+XXXX é esse número em hexadecimal

# os bytes em UTF-8


## 4. Split por espaço

A régua mais simples não usa regex: `texto.split(" ")` corta em cada caractere de espaço.

In [ ]:
reclamacao = mensagens["mensagem"][69]
# corte em cada caractere de espaço


A quebra de linha não é espaço: `"produto.\r\nPrecisa"` vira um token só. Sem argumento,
`split()` corta em qualquer espaço em branco, quebras de linha incluídas:

In [ ]:
# corte em qualquer espaço em branco (espaço, \r, \n)


## 5. Regex: o que é

Um regex é um padrão que descreve um trecho de texto. `\d{1,2}/\d{1,2}` lê-se "um ou dois
dígitos, uma barra, um ou dois dígitos": acha datas como 18/12. Quem cita uma data?

In [ ]:
# marque as mensagens que contêm uma data (o padrão acima)

# as 10 primeiras


Data aparece em reclamação de prazo: "prevista para 18/12", "comprei em 26/03 e até agora não recebi".

## 6. Regex como régua de tokens

Quatro padrões, cada um corrigindo um defeito do anterior:

| padrão | lê-se | defeito |
|---|---|---|
| `\S+` | tudo que não é espaço | a pontuação gruda: "prazo." ≠ "prazo" |
| `\w+` | letras, dígitos e `_` | solta a pontuação, mas parte "R$88,46" em "88" e "46" e perde o emoji |
| `[^\W\d_]+` | só letras (acentuadas também) | perde os números |
| composto | URL, ou dinheiro, ou número, ou palavra, ou emoji | a ordem das alternativas decide: a primeira que casa vence |

In [ ]:
COMPOSTO = (r"https?://\S+|www\.\S+"                      # URL
            r"|R\$\s?\d+(?:[.,]\d+)*"                     # dinheiro
            r"|\d+(?:[.,]\d+)*%?"                         # número
            r"|[^\W\d_]+(?:[-'’][^\W\d_]+)*"              # palavra, com hífen ou apóstrofo
            r"|[\U0001F300-\U0001FAFF☀-➿][\U0001F3FB-\U0001F3FF]?")  # emoji com tom de pele


def tokenizar(texto):
    return re.findall(COMPOSTO, texto)


PADROES = {"\\S+": r"\S+", "\\w+": r"\w+", "só letras": r"[^\W\d_]+", "composto": COMPOSTO}
exemplos = ["pago R$88,46 (2 relógios e frete)", "entrega rápida. Estou satisfeitíssima. 👏🏽"]
for nome, padrao in PADROES.items():
    # os tokens que o padrão tira de cada exemplo


In [ ]:
for nome, padrao in PADROES.items():
    # os tipos (tokens únicos) que o padrão gera no corpus inteiro


## 7. Três tokenizadores pré-treinados

NLTK e spaCy aplicam regras do português escritas por outras pessoas; o tokenizador do GPT
corta em pedaços aprendidos num corpus enorme. Nenhum é treinado aqui.

In [ ]:
import spacy
import tiktoken
from nltk.tokenize import word_tokenize

nlp = spacy.load("pt_core_news_sm", exclude=["tok2vec", "morphologizer", "parser", "lemmatizer",
                                              "attribute_ruler", "ner", "senter"])
gpt = tiktoken.get_encoding("o200k_base")
reguas = {
    "NLTK": lambda t: word_tokenize(t, language="portuguese"),
    "spaCy": lambda t: [tok.text for tok in nlp.tokenizer(t) if not tok.is_space],
    "GPT": lambda t: [gpt.decode([i]) for i in gpt.encode(t)],
}
exemplo = "Embora tenha pago R$88,46 (2 relógios e frete) só recebi um."
for nome, cortar in reguas.items():
    # corte o exemplo e mostre os pedaços


O GPT parte palavras raras em pedaços conhecidos ("relóg", "ios"). Por isso a conta de um
modelo de linguagem é em tokens: o mesmo texto custa mais tokens em português que em inglês.

In [ ]:
pares = [("Não recebi o produto e ninguém responde.", "I did not receive the product and nobody answers."),
         ("Veio com defeito e quero meu dinheiro de volta.", "It came defective and I want my money back.")]
for pt, en in pares:
    # quantos tokens do GPT cada versão gasta


## 8. Estatísticas do corpus

Daqui em diante, a régua é o padrão composto, em minúsculas.

In [ ]:
# cada mensagem tokenizada pelo padrão composto, em minúsculas
# a contagem de cada tipo no corpus inteiro
# quantos tokens tem cada mensagem

print(f"mediana {n_tokens.median():.0f} tokens; 90% têm até {n_tokens.quantile(0.9):.0f}")
print(f"vocabulário: {br(len(contagem))} tipos (tokens únicos)")
contagem.most_common(10)


## 9. Stopwords

A lista do NLTK tira palavras funcionais ("o", "de", "e"), mas também "não". Uma lista
customizada devolve as palavras que têm sentido nas reclamações.

In [ ]:
from nltk.corpus import stopwords

nltk_stop = set(stopwords.words("portuguese"))
COM_SENTIDO = {"não", "nem", "sem", "muito", "mais", "já", "até", "só"}  # negação, falta, intensidade, tempo
# a lista customizada: a do NLTK sem as palavras com sentido

frase = "ainda não recebi o produto".split()
# a frase sem as stopwords de cada lista
# as palavras mais frequentes do corpus, sem as stopwords customizadas


## 10. Zipf

Ordenadas da mais usada para a menos usada, as palavras perdem frequência depressa e sempre
do mesmo jeito, com ou sem stopwords.

In [ ]:
# o ranking: pares (palavra, vezes), da mais usada para a menos usada
# o mesmo ranking sem as stopwords customizadas

for nome, lista in [("com stopwords", ranking), ("sem stopwords", sem_stop)]:
    print(nome)
    # a frequência da 1ª colocada, para comparar
    primeira = lista[0][1]
    for posto in [1, 10, 100, 1000]:
        # a palavra no posto e quantas vezes ela aparece (a lista começa em 0)
        # quantas vezes menos que a 1ª colocada

# só as contagens, na ordem do ranking
# quantos tipos aparecem uma vez só


In [ ]:
# onde começa a cauda: os tipos com 5 ocorrências ou mais vêm antes desse posto
corte = (freqs >= 5).sum()
# cabeça, meio e cauda do ranking
zonas = {"cabeça (100 mais)": freqs[:100], "meio (≥ 5 vezes)": freqs[100:corte], "cauda (< 5 vezes)": freqs[corte:]}
# para cada zona: a parte do texto (ocorrências) e a parte do vocabulário (tipos)
pd.DataFrame({"% do texto": {z: 100 * f.sum() / freqs.sum() for z, f in zonas.items()},
              "% do vocabulário": {z: 100 * len(f) / len(freqs) for z, f in zonas.items()}}).round(1)


A cabeça (stopwords e o assunto geral) faz 62% do texto; a cauda (erros de digitação e
variantes) é três quartos do vocabulário e menos de 4% do texto. As reclamações estão no meio.

## 11. A primeira resposta: as palavras das reclamações

**A pergunta: do que reclamam os clientes?** A primeira resposta são os pares de palavras
seguidas (*bigramas*) muito mais comuns nas avaliações ruins do que nas boas. A medida é a % das
avaliações de cada grupo que contêm o bigrama.

In [ ]:
# os bigramas de cada avaliação, sem repetição
# uma linha por avaliação e bigrama
# em quantas avaliações de cada grupo cada bigrama aparece

# o bigrama tem stopword? (alguma das duas palavras na lista)
# fica com os frequentes (40 avaliações ou mais) e sem stopword

# a % das avaliações ruins e das boas que têm o bigrama
# os 8 mais citados nas ruins, entre os que são 8 vezes (ou mais) mais comuns nas ruins

grafico_ruins_boas(pista.index, pista["ruins"], pista["boas"], "a primeira resposta: bigramas das avaliações ruins")
pista.round(1)


Como ler: "não recebi" está em 14% das avaliações ruins e em 0,3% das boas, cerca de 50 vezes mais.

**Resposta:** a entrega domina. "não recebi", "não chegou", "até agora" e "agora não" dizem que o
produto não chegou ou atrasou.

## 12. O que essa contagem ainda não sabe

"não recebi" e "nao recebi" contam separado, e a mesma avaliação pode entrar mais de uma vez.
É o assunto do próximo laboratório: higiene de corpus.

In [ ]:
# em quantas mensagens aparece cada grafia do bigrama
# quantos review_id aparecem repetidos
